# 🏦 SGA Sales Intelligence & Retention Engine
## Notebook unifié — Audit, Feature Engineering, Segmentation, Recommandation, RL & PDF

Ce notebook remplace les anciens notebooks par une version **cohérente, exécutable, robuste et orientée jury SGA**.

### Objectifs business
1. **Segmentation client** : Standard, Strategic, High-Value, VIP  
2. **Product Propensity / Next Best Action** : CatBoost + similarité sectorielle  
3. **Détection du churn** : alerte proactive basée sur l’évolution récente des flux  
4. **Advisor Feedback Loop** : NLP local + Multi-Armed Bandit (UCB)

### Contraintes respectées
- 100% **local** : Pandas, Scikit-learn, CatBoost, NLTK, ReportLab
- **Aucune API externe**, aucun LLM
- Chaque bloc produit des **insights business explicites**

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import json
import math
import pickle
import re
import unicodedata
from pathlib import Path

import nbformat
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, silhouette_score
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from catboost import CatBoostClassifier

from reportlab.lib import colors
from reportlab.lib.enums import TA_CENTER, TA_LEFT
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
from reportlab.lib.units import cm
from reportlab.pdfgen import canvas
from reportlab.platypus import Paragraph, SimpleDocTemplate, Spacer, Table, TableStyle

try:
    import nltk
    from nltk.corpus import stopwords
    from nltk.tokenize import wordpunct_tokenize
except Exception:
    nltk = None
    stopwords = None
    wordpunct_tokenize = None

sns.set_theme(style="whitegrid")
np.random.seed(42)

SEARCH_DIRS = [Path.cwd(), Path("/mnt/data")]

def find_existing_file(candidates):
    for directory in SEARCH_DIRS:
        for name in candidates:
            path = directory / name
            if path.exists():
                return path
    raise FileNotFoundError(f"Fichier introuvable parmi: {candidates}")

DATA_FILE = find_existing_file(["BDD HACKATHON VF .xlsb (1).xlsx", "BDD HACKATHON VF .xlsb.xlsx", "BDD HACKATHON VF .xlsb", "BDD HACKATHON VF.xlsx"])
FULL_NOTEBOOK = find_existing_file(["AI4SALES_EDA_Full(1).ipynb", "AI4SALES_EDA_Full.ipynb"])
V2_NOTEBOOK = find_existing_file(["AI4SALES_EDA_V2(2).ipynb", "AI4SALES_EDA_V2(1).ipynb", "AI4SALES_EDA_V2.ipynb"])
OUTPUT_DIR = Path("outputs_sga_engine")
OUTPUT_DIR.mkdir(exist_ok=True)

SGA_RED = "#E60028"
SGA_NAVY = "#0B3A67"
SGA_GREY = "#A7A9AC"
SGA_LIGHT = "#F5F7FA"
SGA_GREEN = "#2E8B57"
SGA_GOLD = "#B08D57"

In [ ]:

def clean_col_name(col):
    return re.sub(r"\s+", " ", str(col).strip())

bdd = pd.read_excel(DATA_FILE, sheet_name="BDD")
bdd.columns = [clean_col_name(c) for c in bdd.columns]
bdd["Date"] = pd.to_datetime(bdd["Date"], errors="coerce")

glossaire = pd.read_excel(DATA_FILE, sheet_name="Glossaire", header=None, names=["column", "description"])
glossaire["column"] = glossaire["column"].astype(str).map(clean_col_name)
glossaire["description"] = glossaire["description"].fillna("").astype(str)
glossaire = glossaire.dropna(subset=["column"]).drop_duplicates(subset=["column"]).reset_index(drop=True)

gloss_map = dict(zip(glossaire["column"], glossaire["description"]))

print(f"✅ BDD chargée : {bdd.shape[0]:,} lignes × {bdd.shape[1]} colonnes")
print(f"✅ Clients uniques : {bdd['Client'].nunique():,}")
print(f"✅ Période couverte : {bdd['Date'].min().date()} → {bdd['Date'].max().date()}")
print(f"✅ Nombre de mois distincts : {bdd['Date'].nunique()}")
print("💡 INSIGHT: la base contient bien 15 mois d'historique, ce qui permet une approche proactive plutôt qu'une simple photographie statique.")

In [ ]:

def audit_notebook(notebook_path):
    nb = nbformat.read(notebook_path, as_version=4)
    source = "\n".join(cell.source for cell in nb.cells if cell.cell_type == "code")
    checks = {
        "Chemin de fichier fragile (.xlsb.xlsx)": r"BDD HACKATHON VF \.xlsb\.xlsx",
        "Lecture manuelle openpyxl / iter_rows": r"openpyxl\.load_workbook|iter_rows",
        "Téléchargement NLTK à l'exécution": r"nltk\.download",
        "Usage de KMeans": r"\bKMeans\b",
        "Usage de DBSCAN": r"\bDBSCAN\b",
        "Colonnes potentiellement fuyardes (Chargée/Agence)": r"Chargée de la relation|Agence",
        "Référence fragile à colonne non garantie": r"A_Incident_Payement",
    }
    findings = []
    for label, pattern in checks.items():
        if re.search(pattern, source):
            findings.append(label)
    return findings

audit_rows = []
for path in [FULL_NOTEBOOK, V2_NOTEBOOK]:
    findings = audit_notebook(path)
    for finding in findings:
        audit_rows.append({"Notebook": path.name, "Constat": finding})

audit_df = pd.DataFrame(audit_rows)
display(audit_df)

print("💡 INSIGHT: les anciens notebooks mélangent plusieurs logiques concurrentes (KMeans, DBSCAN, agrégations incohérentes, chargement fragile).")
print("💡 INSIGHT: la nouvelle version corrige quatre points critiques :")
print("   1) suppression des chemins codés en dur")
print("   2) suppression des fuites de variables commerciales dans le ML")
print("   3) remplacement des segmentations incohérentes par un GMM robuste")
print("   4) suppression des dépendances runtime fragiles (ex: nltk.download).")

In [ ]:

pillar_keywords = {
    "Cash Management": [
        "flux créditeurs", "cib", "virement", "versement", "chèque", "remise de chèque", "bdt"
    ],
    "Trade Finance": [
        "flux export", "flux import", "cdi", "sblc", "aval", "credoc", "trade"
    ],
    "Financing/Credit": [
        "découvert", "escompte", "credit", "crédit", "cmt", "leasing",
        "spot", "asf", "caution", "cra", "ocd", "credit-bail"
    ],
    "Risk/Quality": [
        "qualité client", "impay", "irrégulier", "taux d'utilisation",
        "engagement", "creances class", "max_age", "nbr_imp"
    ],
}

pillar_mapping = {pillar: [] for pillar in pillar_keywords}

for col in bdd.columns:
    searchable_text = f"{col} {gloss_map.get(col, '')}".lower()
    for pillar, keywords in pillar_keywords.items():
        if any(keyword in searchable_text for keyword in keywords):
            pillar_mapping[pillar].append({
                "column": col,
                "description": gloss_map.get(col, "")
            })

for pillar, values in pillar_mapping.items():
    print(f"\n🏛️ {pillar} — {len(values)} colonnes repérées")
    preview = pd.DataFrame(values).head(8)
    display(preview)
    print(f"💡 INSIGHT: {pillar} est bien représenté dans la donnée, ce qui permet une lecture 360° du client entreprise.")

In [ ]:
bdd = bdd.sort_values(["Client", "Date"]).reset_index(drop=True)

pdf_guided_product_catalog = [
    {"signal_col": "NB_SGCN_autorisé", "product_label": "SOGECASHNET", "family": "Cash Management", "credit_like": False},
    {"signal_col": "NB carte CIB", "product_label": "Carte CIB Business", "family": "Cash Management", "credit_like": False},
    {"signal_col": "NB_SGATRADE_autorisé", "product_label": "SGATRADE", "family": "Trade Finance", "credit_like": False},
    {"signal_col": "Découvert", "product_label": "Découvert", "family": "Financing/Credit", "credit_like": True},
    {"signal_col": "ESCOMPTE", "product_label": "Escompte", "family": "Financing/Credit", "credit_like": True},
    {"signal_col": "CREDIT_SPOT", "product_label": "Crédit Spot", "family": "Financing/Credit", "credit_like": True},
    {"signal_col": "ASF_ASM", "product_label": "Avance sur Factures", "family": "Financing/Credit", "credit_like": True},
    {"signal_col": "CMT", "product_label": "Crédit Moyen Terme", "family": "Financing/Credit", "credit_like": True},
    {"signal_col": "CAUTIONS_MARCHES", "product_label": "Cautions Marchés", "family": "Financing/Credit", "credit_like": True},
    {"signal_col": "CRA", "product_label": "CRA", "family": "Trade Finance", "credit_like": True},
    {"signal_col": "OCD", "product_label": "OCD", "family": "Trade Finance", "credit_like": True},
    {"signal_col": "AVAL", "product_label": "Aval", "family": "Trade Finance", "credit_like": True},
    {"signal_col": "CDI", "product_label": "Crédit Documentaire Import", "family": "Trade Finance", "credit_like": True},
    {"signal_col": "SBLC", "product_label": "SBLC", "family": "Trade Finance", "credit_like": True},
    {"signal_col": "ENCOURS_LEASING", "product_label": "Leasing Mobilier", "family": "Financing/Credit", "credit_like": True},
    {"signal_col": "CREDIT-BAIL IMMOBILIER", "product_label": "Crédit-Bail Immobilier", "family": "Financing/Credit", "credit_like": True},
    {"signal_col": "NB BDT", "product_label": "Bons du Trésor", "family": "Treasury & Markets", "credit_like": False},
    {"signal_col": "NB FWD autorisé", "product_label": "FX Forward", "family": "Treasury & Markets", "credit_like": False},
    {"signal_col": "NB CVAR utilisé", "product_label": "CVAR", "family": "Treasury & Markets", "credit_like": False},
]

available_product_catalog = [item for item in pdf_guided_product_catalog if item["signal_col"] in bdd.columns]

product_support_rows = []
for item in available_product_catalog:
    series = pd.to_numeric(bdd[item["signal_col"]], errors="coerce").fillna(0)
    client_flag = bdd.groupby("Client")[item["signal_col"]].apply(
        lambda x: int((pd.to_numeric(x, errors="coerce").fillna(0) != 0).any())
    )
    product_support_rows.append({
        **item,
        "rows_non_zero": int((series != 0).sum()),
        "clients_equipped": int(client_flag.sum()),
        "penetration_rate": float(client_flag.mean()),
    })

product_catalog_df = (
    pd.DataFrame(product_support_rows)
      .sort_values(["clients_equipped", "rows_non_zero"], ascending=False)
      .reset_index(drop=True)
)
recommendable_catalog_df = product_catalog_df[product_catalog_df["clients_equipped"] > 0].copy().reset_index(drop=True)
unsupported_catalog_df = product_catalog_df[product_catalog_df["clients_equipped"] == 0].copy().reset_index(drop=True)

product_cols = recommendable_catalog_df["signal_col"].tolist()
credit_product_cols = recommendable_catalog_df.loc[recommendable_catalog_df["credit_like"], "signal_col"].tolist()
non_credit_product_cols = recommendable_catalog_df.loc[~recommendable_catalog_df["credit_like"], "signal_col"].tolist()

product_label_lookup = dict(zip(product_catalog_df["signal_col"], product_catalog_df["product_label"]))
product_family_lookup = dict(zip(product_catalog_df["signal_col"], product_catalog_df["family"]))
product_credit_flag_lookup = dict(zip(product_catalog_df["signal_col"], product_catalog_df["credit_like"]))

client_product_flags_raw = (
    bdd.groupby("Client")[product_cols]
       .apply(lambda frame: (frame.fillna(0) != 0).any().astype(int))
)

missing_pct = (bdd.isna().mean() * 100).sort_values(ascending=False).head(20)
pnb = pd.to_numeric(bdd["PNB NET"], errors="coerce")
pnb_clip = pnb.clip(pnb.quantile(0.01), pnb.quantile(0.99))
monthly_flux = bdd.groupby("Date")["Flux Créditeurs"].sum()
monthly_active_clients = (
    bdd.assign(active=(pd.to_numeric(bdd["Flux Créditeurs"], errors="coerce").fillna(0) > 0).astype(int))
       .groupby("Date")["Client"].nunique()
)

penetration = (client_product_flags_raw.mean().sort_values(ascending=False) * 100)
penetration_named = penetration.copy()
penetration_named.index = penetration_named.index.map(lambda x: product_label_lookup.get(x, x))

display(product_catalog_df[["product_label", "signal_col", "family", "credit_like", "clients_equipped", "penetration_rate"]])

if len(unsupported_catalog_df):
    display(unsupported_catalog_df[["product_label", "signal_col", "family", "clients_equipped"]])

fig, axes = plt.subplots(3, 2, figsize=(18, 16))
axes = axes.flatten()

missing_pct.sort_values().plot(kind="barh", ax=axes[0], color=SGA_GREY)
axes[0].set_title("1) Top colonnes les plus manquantes", color=SGA_NAVY, weight="bold")
axes[0].set_xlabel("% de valeurs manquantes")

axes[1].hist(pnb_clip.dropna(), bins=40, color=SGA_RED, edgecolor="white")
axes[1].set_title("2) Distribution PNB NET (winsorisée)", color=SGA_NAVY, weight="bold")
axes[1].set_xlabel("PNB NET")
axes[1].set_ylabel("Nombre de lignes")

axes[2].boxplot(pnb.dropna().clip(lower=pnb.quantile(0.01), upper=pnb.quantile(0.99)))
axes[2].set_title("3) Outliers PNB NET", color=SGA_NAVY, weight="bold")
axes[2].set_ylabel("PNB NET")

axes[3].plot(monthly_flux.index, monthly_flux.values, marker="o", color=SGA_GREEN, linewidth=2)
axes[3].set_title("4) Flux créditeurs agrégés par mois", color=SGA_NAVY, weight="bold")
axes[3].tick_params(axis="x", rotation=45)
axes[3].set_ylabel("Flux total")

penetration_named.head(15).sort_values().plot(kind="barh", ax=axes[4], color=SGA_GOLD)
axes[4].set_title("5) Pénétration produits (univers complet)", color=SGA_NAVY, weight="bold")
axes[4].set_xlabel("% clients équipés")

axes[5].plot(monthly_active_clients.index, monthly_active_clients.values, marker="o", color=SGA_NAVY, linewidth=2)
axes[5].set_title("6) Clients actifs par mois", color=SGA_NAVY, weight="bold")
axes[5].tick_params(axis="x", rotation=45)
axes[5].set_ylabel("Nb clients")

plt.tight_layout()
eda_path = OUTPUT_DIR / "dirty_eda_overview.png"
plt.savefig(eda_path, dpi=160, bbox_inches="tight")
plt.show()

print(f"✅ Figure EDA sauvegardée : {eda_path.name}")
print(f"✅ Univers produit piloté par le PDF + BDD : {len(product_catalog_df)} produits/services identifiés, dont {len(recommendable_catalog_df)} réellement observés dans la donnée.")
print(f"💡 INSIGHT: {missing_pct.index[0]} et {missing_pct.index[1]} sont quasi inexploitables tels quels → nettoyage obligatoire avant modélisation.")
print("💡 INSIGHT: la distribution du PNB NET est très asymétrique → scaler robuste et clipping nécessaires pour éviter des clusters artificiels.")
print("💡 INSIGHT: la pénétration produit est fortement déséquilibrée → CatBoost doit être entraîné produit par produit, avec fallback sectoriel pour les produits rares.")
print("💡 INSIGHT: le portefeuille exploitable ne se limite pas au découvert / spot / caution ; il couvre aussi les services cash, trade et marchés observés dans la BDD.")
print("💡 INSIGHT: les produits présents dans le PDF mais absents de la donnée active restent catalogués pour la roadmap métier, mais ne sont pas recommandés sans signal historique.")

In [ ]:
def safe_numeric(series):
    return pd.to_numeric(series, errors="coerce")

def safe_last_non_null(series):
    series = series.dropna()
    return series.iloc[-1] if len(series) else np.nan

def safe_nonzero_any(series):
    return int((safe_numeric(series).fillna(0) != 0).any())

def safe_positive_months(series):
    return int((safe_numeric(series).fillna(0) > 0).sum())

def safe_nonzero_months(series):
    return int((safe_numeric(series).fillna(0) != 0).sum())

def safe_divide(numerator, denominator, default=0.0):
    denominator = np.where(np.abs(denominator) < 1e-12, np.nan, denominator)
    result = numerator / denominator
    result = np.where(np.isnan(result), default, result)
    return result

def pdf_safe(text):
    return unicodedata.normalize("NFKD", str(text)).encode("ascii", "ignore").decode("ascii")

def slugify_text(text):
    text = pdf_safe(text).lower()
    text = re.sub(r"[^a-z0-9]+", "_", text).strip("_")
    return text or "unknown"

dates_sorted = sorted(pd.Series(bdd["Date"].dropna().unique()).tolist())
current_period = dates_sorted[-3:]
previous_period = dates_sorted[-6:-3]

current_mask = bdd["Date"].isin(current_period)
previous_mask = bdd["Date"].isin(previous_period)

digital_cols = [c for c in ["NB_SGCN_utilisé", "NB_SGCN_autorisé", "NB_SGATRADE_autorisé"] if c in bdd.columns]
risk_cols = [
    c for c in [
        "ESCOMPTE EFFETS IMPAYES", "IMP_SPOT_ASF", "IMP_CMT", "IMP_CAUTIONS_MISE_EN_JEU",
        "IMPAYES_LEASING", "IMP INT CREDIT BAIL IMMO", "CREANCES_CLASSEES_B",
        "INT_ACC_CREANCES_CLASSEES_B", "CREANCES_CLASSEES_HB", "nbr_imp_total", "max_age"
    ] if c in bdd.columns
]
transaction_cols = [
    c for c in [
        "Chèque de banque.C", "Effet.C", "Versement.C", "VirementC.",
        "Change et frais de missions.D", "Chèque de banque.D", "Effet.D",
        "Remise de chèque.D", "Retrait.D", "Versement.D"
    ] if c in bdd.columns
]
static_cols = [
    c for c in [
        "Qualité Client", "Chargée de la relation", "Segmenattion", "client Actif/Inactif",
        "Client non Résident OUI/NON", "Agence", "Marché", "Secteur d'activité",
        "Secteur d'activité détaillé", "Groupe", "Date de création", "chiffre d'affaire",
        "Nbr d'employés", "Nbr Actionnaire", "Wilaya", "Ancienneté de la relation"
    ] if c in bdd.columns
]

In [ ]:
group = bdd.groupby("Client", sort=False)
latest_rows = bdd.groupby("Client", sort=False).tail(1).set_index("Client")

df_final_features = pd.DataFrame(index=sorted(bdd["Client"].dropna().unique()))
df_final_features.index.name = "Client"

for col in static_cols:
    df_final_features[col] = latest_rows[col]

df_final_features["Flux_Crediteurs_Current_3M"] = bdd.loc[current_mask].groupby("Client")["Flux Créditeurs"].sum()
df_final_features["Flux_Crediteurs_Previous_3M"] = bdd.loc[previous_mask].groupby("Client")["Flux Créditeurs"].sum()
df_final_features[["Flux_Crediteurs_Current_3M", "Flux_Crediteurs_Previous_3M"]] = (
    df_final_features[["Flux_Crediteurs_Current_3M", "Flux_Crediteurs_Previous_3M"]].fillna(0)
)

evolution_ratio = safe_divide(
    df_final_features["Flux_Crediteurs_Current_3M"].values,
    df_final_features["Flux_Crediteurs_Previous_3M"].replace(0, np.nan).values,
    default=np.nan,
)
df_final_features["Taux_Evolution_Flux"] = evolution_ratio

mask_prev_zero = df_final_features["Flux_Crediteurs_Previous_3M"].eq(0)
mask_curr_positive = df_final_features["Flux_Crediteurs_Current_3M"].gt(0)
mask_both_zero = mask_prev_zero & df_final_features["Flux_Crediteurs_Current_3M"].eq(0)

df_final_features.loc[mask_prev_zero & mask_curr_positive, "Taux_Evolution_Flux"] = 999.0
df_final_features.loc[mask_both_zero, "Taux_Evolution_Flux"] = 1.0
df_final_features["Taux_Evolution_Flux"] = (
    pd.to_numeric(df_final_features["Taux_Evolution_Flux"], errors="coerce")
      .fillna(1.0)
      .clip(lower=0, upper=999)
)
df_final_features["Churn_Alert_Flag"] = (df_final_features["Taux_Evolution_Flux"] < 0.7).astype(int)

agg_sum_cols = [c for c in ["Flux Créditeurs", "Flux Export", "Flux IMPORT", "PNB NET", "PNB Brut", "Commissions", "MI", "total engagement"] if c in bdd.columns]
for col in agg_sum_cols:
    df_final_features[f"{col}_sum_15m"] = group[col].sum(min_count=1).fillna(0)
    df_final_features[f"{col}_mean_15m"] = group[col].mean().fillna(0)
    df_final_features[f"{col}_last"] = group[col].apply(safe_last_non_null)

for col in digital_cols + risk_cols + transaction_cols:
    df_final_features[f"{col}_sum_15m"] = group[col].sum(min_count=1).fillna(0)
    df_final_features[f"{col}_last"] = group[col].apply(safe_last_non_null)

for col in product_cols:
    df_final_features[f"has_{col}"] = group[col].apply(safe_nonzero_any)
    df_final_features[f"{col}_sum_15m"] = group[col].sum(min_count=1).fillna(0)
    df_final_features[f"{col}_max_15m"] = group[col].max().fillna(0)

df_final_features["Months_Observed"] = group["Date"].nunique()
df_final_features["Months_Active_Flux"] = group["Flux Créditeurs"].apply(safe_positive_months)
df_final_features["Months_Active_PNB"] = group["PNB NET"].apply(safe_nonzero_months)
df_final_features["Nb_Produits_Detenus"] = df_final_features[[f"has_{c}" for c in product_cols]].sum(axis=1)
df_final_features["Nb_Produits_Credit"] = (
    df_final_features[[f"has_{c}" for c in credit_product_cols]].sum(axis=1)
    if credit_product_cols else 0
)
df_final_features["Nb_Produits_Non_Credit"] = (
    df_final_features[[f"has_{c}" for c in non_credit_product_cols]].sum(axis=1)
    if non_credit_product_cols else 0
)

if "Taux d'utilisation M_last" in df_final_features.columns:
    df_final_features["Taux_Utilisation_Credit"] = pd.to_numeric(
        df_final_features["Taux d'utilisation M_last"], errors="coerce"
    )
else:
    df_final_features["Taux_Utilisation_Credit"] = np.nan

if df_final_features["Taux_Utilisation_Credit"].isna().all():
    if "ENG (sans : CMT, LEA, DCP) M" in bdd.columns and "Auto (sans : CMT, LEA, DCP) M" in bdd.columns:
        numer = group["ENG (sans : CMT, LEA, DCP) M"].apply(safe_last_non_null)
        denom = group["Auto (sans : CMT, LEA, DCP) M"].apply(safe_last_non_null).replace(0, np.nan)
        df_final_features["Taux_Utilisation_Credit"] = numer / denom

df_final_features["Taux_Utilisation_Credit"] = (
    pd.to_numeric(df_final_features["Taux_Utilisation_Credit"], errors="coerce")
      .fillna(0)
      .clip(lower=0)
)

if digital_cols:
    digital_latest = latest_rows[digital_cols].apply(pd.to_numeric, errors="coerce").fillna(0)
    df_final_features["Score_Digital_Actif"] = (digital_latest.sum(axis=1) > 0).astype(int)
else:
    df_final_features["Score_Digital_Actif"] = 0

credit_flag_columns = [f"has_{c}" for c in credit_product_cols]
if credit_flag_columns:
    df_final_features["Existing_Credit_Line"] = (
        (df_final_features[credit_flag_columns].sum(axis=1) > 0)
        | (df_final_features["Taux_Utilisation_Credit"] > 0)
    ).astype(int)
else:
    df_final_features["Existing_Credit_Line"] = (df_final_features["Taux_Utilisation_Credit"] > 0).astype(int)

df_final_features["No_New_Credit_Allowed"] = (
    (df_final_features["Existing_Credit_Line"] == 1)
    & (df_final_features["Taux_Utilisation_Credit"] < 0.3)
).astype(int)

df_final_features["PNB_NET_15m"] = pd.to_numeric(df_final_features["PNB NET_sum_15m"], errors="coerce").fillna(0)
df_final_features["Flux_Crediteurs_15m"] = pd.to_numeric(df_final_features["Flux Créditeurs_sum_15m"], errors="coerce").fillna(0)
df_final_features["Risk_Impayes_Total"] = 0.0
for col in risk_cols:
    derived = [c for c in df_final_features.columns if c.startswith(col)]
    for derived_col in derived:
        df_final_features["Risk_Impayes_Total"] += pd.to_numeric(df_final_features[derived_col], errors="coerce").fillna(0)

df_final_features["Is_International"] = (
    (
        pd.to_numeric(df_final_features.get("Flux Export_sum_15m", 0), errors="coerce").fillna(0)
        + pd.to_numeric(df_final_features.get("Flux IMPORT_sum_15m", 0), errors="coerce").fillna(0)
    ) != 0
).astype(int)

transaction_sum_cols = [c for c in df_final_features.columns if c.endswith("_sum_15m") and any(token in c for token in ["Virement", "Versement", "Retrait", "Remise de chèque", "Effet", "Chèque de banque"])]
df_final_features["Cash_Transactions_15m"] = df_final_features[transaction_sum_cols].sum(axis=1) if transaction_sum_cols else 0

df_final_features["Qualité Client"] = pd.to_numeric(df_final_features["Qualité Client"], errors="coerce")
df_final_features["chiffre d'affaire"] = pd.to_numeric(df_final_features["chiffre d'affaire"], errors="coerce")
df_final_features["Nbr d'employés"] = pd.to_numeric(df_final_features["Nbr d'employés"], errors="coerce")

df_final_features["Client_Size"] = pd.cut(
    df_final_features["chiffre d'affaire"].fillna(-1),
    bins=[-np.inf, 0, 1e8, 5e8, 1e9, np.inf],
    labels=["Unknown", "Small", "Mid", "Large", "Enterprise"]
).astype(str)

fallback_mask = (df_final_features["Client_Size"] == "Unknown") & df_final_features["Nbr d'employés"].notna()
df_final_features.loc[fallback_mask, "Client_Size"] = pd.cut(
    df_final_features.loc[fallback_mask, "Nbr d'employés"],
    bins=[0, 20, 100, 250, np.inf],
    labels=["Small", "Mid", "Large", "Enterprise"]
).astype(str)

if "Date de création" in df_final_features.columns:
    df_final_features["Date de création"] = pd.to_datetime(df_final_features["Date de création"], errors="coerce")
    df_final_features["Anciennete_Client_Jours"] = (
        pd.Timestamp(max(dates_sorted)) - df_final_features["Date de création"]
    ).dt.days.clip(lower=0)

if "Secteur d'activité" in df_final_features.columns:
    df_final_features["Secteur d'activité"] = df_final_features["Secteur d'activité"].fillna("Inconnu")
    sector_mean_pnb = df_final_features.groupby("Secteur d'activité")["PNB_NET_15m"].transform("mean")
    df_final_features["Ecart_Moyenne_Secteur"] = df_final_features["PNB_NET_15m"] - sector_mean_pnb
else:
    df_final_features["Ecart_Moyenne_Secteur"] = 0

df_final_features = df_final_features.replace([np.inf, -np.inf], np.nan)
numeric_cols_final = df_final_features.select_dtypes(include=[np.number]).columns
df_final_features[numeric_cols_final] = df_final_features[numeric_cols_final].fillna(0)

print(f"✅ Dataset aplati : {df_final_features.shape[0]:,} clients × {df_final_features.shape[1]} features")
print(f"💡 INSIGHT: {df_final_features['Churn_Alert_Flag'].mean():.1%} des clients déclenchent une alerte churn selon la règle SGA (< 0.7).")
print(f"💡 INSIGHT: {df_final_features['No_New_Credit_Allowed'].mean():.1%} des clients sont filtrés anti-spam uniquement sur les produits de crédit/financement.")
print(f"💡 INSIGHT: en moyenne, un client détient {df_final_features['Nb_Produits_Detenus'].mean():.2f} produits/services observés dans l'univers enrichi.")
display(df_final_features.head(3))

In [ ]:
sector_col = "Secteur d'activité"
product_flag_cols = [f"has_{c}" for c in product_cols]
sector_product_penetration = df_final_features.groupby(sector_col)[product_flag_cols].mean()

def top_missing_products(row, top_n=3):
    sector_value = row.get(sector_col, "Inconnu")
    if sector_value not in sector_product_penetration.index:
        return []
    ranking = sector_product_penetration.loc[sector_value].sort_values(ascending=False)
    missing = []
    for flag_name, score in ranking.items():
        product_signal = flag_name.replace("has_", "", 1)
        if row.get(flag_name, 0) == 0 and score > 0:
            missing.append({
                "product_signal": product_signal,
                "product": product_label_lookup.get(product_signal, product_signal),
                "family": product_family_lookup.get(product_signal, "Other"),
                "sector_penetration": round(float(score), 4)
            })
        if len(missing) >= top_n:
            break
    return missing

missing_products = df_final_features.apply(top_missing_products, axis=1)
for rank in [1, 2, 3]:
    df_final_features[f"Produit_Manquant_Secteur_{rank}"] = missing_products.map(
        lambda x: x[rank - 1]["product"] if len(x) >= rank else ""
    )

df_final_features["Produit_Manquant_Secteur_JSON"] = missing_products.map(lambda x: json.dumps(x, ensure_ascii=False))

most_missing = df_final_features["Produit_Manquant_Secteur_1"].replace("", np.nan).value_counts().head(10)
plt.figure(figsize=(10, 5))
most_missing.sort_values().plot(kind="barh", color=SGA_NAVY)
plt.title("Produits look-alike les plus souvent manquants", color=SGA_NAVY, weight="bold")
plt.xlabel("Nombre de clients")
plt.tight_layout()
lookalike_plot = OUTPUT_DIR / "lookalike_missing_products.png"
plt.savefig(lookalike_plot, dpi=160, bbox_inches="tight")
plt.show()

print("💡 INSIGHT: la logique look-alike sectorielle permet de transformer un portefeuille 'muet' en portefeuille actionnable sur l'ensemble des familles produits.")
print(f"💡 INSIGHT: le produit sectoriel manquant le plus fréquent est '{most_missing.index[-1] if len(most_missing) else 'N/A'}' parmi les top écarts.")
print("💡 INSIGHT: chaque client dispose désormais d'un Top 3 de produits manquants du profil sectoriel, et non plus d'un simple Top 2.")

In [ ]:

cluster_features = [
    c for c in [
        "PNB_NET_15m", "Flux_Crediteurs_15m", "Flux Export_sum_15m", "Flux IMPORT_sum_15m",
        "total engagement_mean_15m", "Nb_Produits_Detenus", "Taux_Utilisation_Credit",
        "Risk_Impayes_Total", "Score_Digital_Actif", "Is_International",
        "Months_Active_Flux", "Qualité Client", "Ecart_Moyenne_Secteur",
        "Cash_Transactions_15m"
    ] if c in df_final_features.columns
]

X_cluster = df_final_features[cluster_features].copy()
for col in X_cluster.columns:
    X_cluster[col] = pd.to_numeric(X_cluster[col], errors="coerce")
    q01, q99 = X_cluster[col].quantile([0.01, 0.99])
    X_cluster[col] = X_cluster[col].clip(q01, q99)

X_cluster = X_cluster.fillna(X_cluster.median())
cluster_scaler = RobustScaler()
X_cluster_scaled = cluster_scaler.fit_transform(X_cluster)

gmm = GaussianMixture(n_components=4, covariance_type="full", random_state=42, reg_covar=1e-6)
gmm_labels = gmm.fit_predict(X_cluster_scaled)
df_final_features["Persona_Cluster"] = gmm_labels

cluster_profile = df_final_features.groupby("Persona_Cluster")[cluster_features].median()

cluster_norm = (cluster_profile - cluster_profile.min()) / (cluster_profile.max() - cluster_profile.min() + 1e-9)
value_score = (
    0.45 * cluster_norm["PNB_NET_15m"]
    + 0.25 * cluster_norm["Flux_Crediteurs_15m"]
    + 0.15 * cluster_norm["Nb_Produits_Detenus"]
    + 0.10 * cluster_norm.get("total engagement_mean_15m", 0)
    - 0.05 * cluster_norm["Risk_Impayes_Total"]
)
strategy_score = (
    0.40 * cluster_norm["Score_Digital_Actif"]
    + 0.35 * cluster_norm["Is_International"]
    + 0.15 * cluster_norm["Nb_Produits_Detenus"]
    + 0.10 * cluster_norm["Cash_Transactions_15m"]
)

vip_cluster = value_score.idxmax()
standard_cluster = value_score.idxmin()
remaining_clusters = [c for c in cluster_profile.index if c not in [vip_cluster, standard_cluster]]
strategic_cluster = max(remaining_clusters, key=lambda c: strategy_score.loc[c])
high_value_cluster = [c for c in remaining_clusters if c != strategic_cluster][0]

persona_map = {
    standard_cluster: "Standard",
    strategic_cluster: "Strategic",
    high_value_cluster: "High-Value",
    vip_cluster: "VIP",
}

df_final_features["Persona_Name"] = df_final_features["Persona_Cluster"].map(persona_map)

persona_counts = df_final_features["Persona_Name"].value_counts()

plt.figure(figsize=(8, 5))
persona_counts.reindex(["Standard", "Strategic", "High-Value", "VIP"]).plot(kind="bar", color=[SGA_GREY, SGA_NAVY, SGA_GOLD, SGA_RED])
plt.title("Distribution des 4 personas GMM", color=SGA_NAVY, weight="bold")
plt.ylabel("Nombre de clients")
plt.xticks(rotation=0)
plt.tight_layout()
cluster_plot = OUTPUT_DIR / "gmm_personas_distribution.png"
plt.savefig(cluster_plot, dpi=160, bbox_inches="tight")
plt.show()

display(cluster_profile.round(2))
print("💡 INSIGHT: GMM permet une segmentation plus crédible que K-Means/DBSCAN sur ce type de base hétérogène.")
print(f"💡 INSIGHT: le cluster VIP représente {persona_counts.get('VIP', 0):,} clients à forte valeur, alors que le cluster Standard concentre le volume.")

In [ ]:
product_penetration = df_final_features[product_flag_cols].mean().sort_values(ascending=False)

model_numeric_cols = [
    c for c in [
        "PNB_NET_15m", "Flux_Crediteurs_15m", "Flux Export_sum_15m", "Flux IMPORT_sum_15m",
        "total engagement_mean_15m", "Taux_Utilisation_Credit", "Score_Digital_Actif",
        "Risk_Impayes_Total", "Months_Active_Flux", "Qualité Client",
        "Ecart_Moyenne_Secteur", "Nb_Produits_Detenus", "Cash_Transactions_15m",
        "chiffre d'affaire", "Nbr d'employés", "Anciennete_Client_Jours",
        "Nb_Produits_Credit", "Nb_Produits_Non_Credit"
    ] if c in df_final_features.columns
]
model_categorical_cols = [
    c for c in [
        "Secteur d'activité", "Secteur d'activité détaillé", "Wilaya", "Segmenattion",
        "Marché", "Client_Size", "Persona_Name", "client Actif/Inactif",
        "Client non Résident OUI/NON"
    ] if c in df_final_features.columns
]

X_model_base = df_final_features[model_numeric_cols + model_categorical_cols].copy()
for col in model_numeric_cols:
    X_model_base[col] = pd.to_numeric(X_model_base[col], errors="coerce").fillna(X_model_base[col].median())
for col in model_categorical_cols:
    X_model_base[col] = X_model_base[col].fillna("Inconnu").astype(str)

content_cols = [c for c in ["Wilaya", "Secteur d'activité", "Client_Size", "Segmenattion"] if c in df_final_features.columns]
content_frame = df_final_features[content_cols].fillna("Inconnu").astype(str)

content_encoder = OneHotEncoder(handle_unknown="ignore")
content_matrix = content_encoder.fit_transform(content_frame)

n_neighbors = min(12, len(df_final_features))
content_knn = NearestNeighbors(metric="cosine", algorithm="brute", n_neighbors=n_neighbors)
content_knn.fit(content_matrix)
content_distances, content_indices = content_knn.kneighbors(content_matrix)

product_models = {}
model_metrics = {}
recommendation_scores = pd.DataFrame(index=df_final_features.index)
min_positive_for_catboost = 60

def weighted_neighbor_score(client_position, product_name):
    neighbors = content_indices[client_position][1:]
    distances = content_distances[client_position][1:]
    if len(neighbors) == 0:
        return 0.0
    neighbor_flags = df_final_features.iloc[neighbors][f"has_{product_name}"].values.astype(float)
    weights = np.clip(1 - distances, 0, 1)
    if weights.sum() == 0:
        return float(neighbor_flags.mean())
    return float(np.average(neighbor_flags, weights=weights))

def sector_prior_score(client_id, product_name):
    sector_value = df_final_features.loc[client_id, sector_col] if sector_col in df_final_features.columns else "Inconnu"
    if sector_value in sector_product_penetration.index:
        return float(sector_product_penetration.loc[sector_value, f"has_{product_name}"])
    return float(product_penetration.get(f"has_{product_name}", 0.0))

for product_name in product_cols:
    y = df_final_features[f"has_{product_name}"].astype(int)
    X_product = X_model_base.copy()

    leak_like_cols = [c for c in X_product.columns if product_name.lower() in c.lower()]
    leak_like_cols += [
        c for c in X_product.columns
        if product_label_lookup.get(product_name, product_name).lower() in c.lower()
    ]
    X_product = X_product.drop(columns=list(set(leak_like_cols)), errors="ignore")

    categorical_indices = [
        X_product.columns.get_loc(col)
        for col in X_product.select_dtypes(include="object").columns
    ]

    strategy = "fallback"
    auc_test = None
    if y.nunique() >= 2 and int(y.sum()) >= min_positive_for_catboost:
        X_train, X_test, y_train, y_test = train_test_split(
            X_product, y, test_size=0.2, random_state=42, stratify=y
        )

        model = CatBoostClassifier(
            iterations=35,
            depth=4,
            learning_rate=0.07,
            loss_function="Logloss",
            eval_metric="AUC",
            verbose=False,
            random_seed=42,
            auto_class_weights="Balanced"
        )
        model.fit(X_train, y_train, cat_features=categorical_indices)

        test_proba = model.predict_proba(X_test)[:, 1]
        full_proba = model.predict_proba(X_product)[:, 1]

        auc_test = float(roc_auc_score(y_test, test_proba))
        strategy = "catboost+content+sector"
        product_models[product_name] = {
            "model": model,
            "feature_columns": X_product.columns.tolist()
        }
        recommendation_scores[f"model_{product_name}"] = full_proba
    else:
        product_models[product_name] = None
        recommendation_scores[f"model_{product_name}"] = float(y.mean())

    content_scores = [weighted_neighbor_score(i, product_name) for i in range(len(df_final_features))]
    sector_scores = [sector_prior_score(client_id, product_name) for client_id in df_final_features.index]

    recommendation_scores[f"content_{product_name}"] = content_scores
    recommendation_scores[f"sector_{product_name}"] = sector_scores

    if product_models[product_name] is not None:
        recommendation_scores[f"blended_{product_name}"] = (
            0.55 * recommendation_scores[f"model_{product_name}"]
            + 0.25 * recommendation_scores[f"content_{product_name}"]
            + 0.20 * recommendation_scores[f"sector_{product_name}"]
        )
    else:
        recommendation_scores[f"blended_{product_name}"] = (
            0.60 * recommendation_scores[f"content_{product_name}"]
            + 0.40 * recommendation_scores[f"sector_{product_name}"]
        )

    recommendation_scores.loc[df_final_features[f"has_{product_name}"] == 1, f"blended_{product_name}"] = 0.0
    if product_credit_flag_lookup.get(product_name, False):
        recommendation_scores.loc[
            df_final_features["No_New_Credit_Allowed"] == 1,
            f"blended_{product_name}"
        ] = 0.0

    model_metrics[product_name] = {
        "product_label": product_label_lookup.get(product_name, product_name),
        "family": product_family_lookup.get(product_name, "Other"),
        "auc_test": auc_test,
        "positive_rate": float(y.mean()),
        "positives": int(y.sum()),
        "strategy": strategy,
        "credit_like": bool(product_credit_flag_lookup.get(product_name, False)),
    }

def recommend_products_for_client(client_id, top_n=3):
    if client_id not in df_final_features.index:
        return []
    scored = []
    for product_name in product_cols:
        score = float(recommendation_scores.loc[client_id, f"blended_{product_name}"])
        if score <= 0:
            continue
        scored.append({
            "product_signal": product_name,
            "product": product_label_lookup.get(product_name, product_name),
            "family": product_family_lookup.get(product_name, "Other"),
            "score": round(score, 4),
            "model_score": round(float(recommendation_scores.loc[client_id, f"model_{product_name}"]), 4),
            "content_score": round(float(recommendation_scores.loc[client_id, f"content_{product_name}"]), 4),
            "sector_score": round(float(recommendation_scores.loc[client_id, f"sector_{product_name}"]), 4),
            "has_model": int(product_models[product_name] is not None),
        })
    scored = sorted(scored, key=lambda x: (x["score"], x["sector_score"], x["content_score"]), reverse=True)
    return scored[:top_n]

model_metrics_df = (
    pd.DataFrame(model_metrics).T
      .sort_values(["positives", "positive_rate"], ascending=False)
      .reset_index()
      .rename(columns={"index": "product_signal"})
)
display(model_metrics_df)

sample_clients = df_final_features.sort_values(["Churn_Alert_Flag", "PNB_NET_15m"], ascending=[False, False]).head(5).index.tolist()
sample_recos = {client_id: recommend_products_for_client(client_id, top_n=3) for client_id in sample_clients}
display(pd.DataFrame({"Client": list(sample_recos.keys()), "Recommendations": [json.dumps(v, ensure_ascii=False) for v in sample_recos.values()]}))

print(f"✅ Produits recommandables scorés : {len(product_cols)}")
print(f"✅ Produits avec modèle CatBoost complet : {sum(v is not None for v in product_models.values())}")
print("💡 INSIGHT: le moteur calcule désormais les scores sur l'univers complet des produits observés, puis retourne le Top 3 des produits manquants du profil du client.")
print("💡 INSIGHT: le score final mélange CatBoost, similarité de contenu et prior sectoriel ; les produits rares restent éligibles via fallback business plutôt que d'être ignorés.")
print("💡 INSIGHT: le filtre anti-spam ne coupe que les produits de crédit/financement sous-utilisés, sans bloquer les recommandations cash, trade ou marchés.")

In [ ]:

LOCAL_FRENCH_STOPWORDS = {
    "de", "la", "le", "les", "un", "une", "du", "des", "et", "ou", "en", "dans", "pour",
    "sur", "au", "aux", "ce", "cet", "cette", "ces", "est", "sont", "avec", "par", "pas",
    "plus", "moins", "tres", "très", "ne", "que", "qui", "quoi", "dont", "a", "ai", "avons",
    "vous", "nous", "il", "elle", "ils", "elles", "je", "tu", "mais", "car", "comme"
}

POSITIVE_BANKING_WORDS = {
    "accepte", "accepté", "adopte", "adopté", "utile", "pertinent", "opportun", "satisfait",
    "rentable", "intéressant", "interessant", "rapide", "fluide", "confiance", "favorable"
}
NEGATIVE_BANKING_WORDS = {
    "refus", "refuse", "refusé", "cher", "risque", "bloque", "bloqué", "plaintes", "plainte",
    "inutile", "retard", "problème", "probleme", "impayé", "impaye", "insatisfait", "non"
}

def get_french_stopwords():
    if stopwords is not None:
        try:
            return set(stopwords.words("french"))
        except Exception:
            return LOCAL_FRENCH_STOPWORDS
    return LOCAL_FRENCH_STOPWORDS

FRENCH_STOPWORDS = get_french_stopwords()

def tokenize_french_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-ZÀ-ÿ0-9\s\-']", " ", text)
    if wordpunct_tokenize is not None:
        tokens = wordpunct_tokenize(text)
    else:
        tokens = re.findall(r"[a-zA-ZÀ-ÿ0-9']+", text)
    return [tok for tok in tokens if tok not in FRENCH_STOPWORDS and len(tok) > 1]

def score_feedback_sentiment(comment):
    tokens = tokenize_french_text(comment)
    pos_hits = sum(tok in POSITIVE_BANKING_WORDS for tok in tokens)
    neg_hits = sum(tok in NEGATIVE_BANKING_WORDS for tok in tokens)
    if pos_hits > neg_hits:
        sentiment = "positive"
        multiplier = 1.25
    elif neg_hits > pos_hits:
        sentiment = "negative"
        multiplier = 0.75
    else:
        sentiment = "neutral"
        multiplier = 1.00
    return {
        "tokens": tokens,
        "positive_hits": pos_hits,
        "negative_hits": neg_hits,
        "sentiment": sentiment,
        "sentiment_multiplier": multiplier
    }

class UCBBandit:
    def __init__(self, arms, contexts=None):
        self.arms = list(arms)
        self.contexts = list(contexts) if contexts is not None else ["GLOBAL"]
        self.counts = {ctx: {arm: 0 for arm in self.arms} for ctx in self.contexts}
        self.values = {ctx: {arm: 0.0 for arm in self.arms} for ctx in self.contexts}

    def _ctx(self, context):
        return context if context in self.contexts else "GLOBAL"

    def recommend(self, context="GLOBAL"):
        ctx = self._ctx(context)
        total_counts = sum(self.counts[ctx].values()) + 1
        scores = {}
        for arm in self.arms:
            arm_count = self.counts[ctx][arm]
            if arm_count == 0:
                scores[arm] = float("inf")
            else:
                bonus = math.sqrt(2 * math.log(total_counts) / arm_count)
                scores[arm] = self.values[ctx][arm] + bonus
        return max(scores, key=scores.get), scores

    def update(self, arm, reward, context="GLOBAL"):
        ctx = self._ctx(context)
        self.counts[ctx][arm] += 1
        n = self.counts[ctx][arm]
        old_value = self.values[ctx][arm]
        self.values[ctx][arm] = old_value + (reward - old_value) / n

def compute_reward(accepted, comment):
    sentiment_info = score_feedback_sentiment(comment)
    base = 1 if accepted else -1
    reward = base * sentiment_info["sentiment_multiplier"]
    sentiment_info["reward"] = reward
    return sentiment_info

top_products = ["Découvert", "ESCOMPTE", "CREDIT_SPOT", "CMT", "CDI", "SBLC", "ENCOURS_LEASING"]
bandit_contexts = sorted(df_final_features["Persona_Name"].dropna().unique().tolist())
bandit = UCBBandit(arms=top_products, contexts=bandit_contexts if bandit_contexts else ["GLOBAL"])

feedback_demo = [
    {"persona": "Standard", "product": top_products[0], "accepted": True,  "comment": "Proposition utile et pertinente, client satisfait"},
    {"persona": "VIP",      "product": top_products[1], "accepted": False, "comment": "Produit jugé trop cher et peu opportun"},
    {"persona": "Strategic","product": top_products[2], "accepted": True,  "comment": "Adopte rapidement, bon timing commercial"},
]

feedback_rows = []
for row in feedback_demo:
    info = compute_reward(row["accepted"], row["comment"])
    context_value = row["persona"] if row["persona"] in bandit.contexts else bandit.contexts[0]
    bandit.update(row["product"], info["reward"], context=context_value)
    feedback_rows.append({
        **row,
        "sentiment": info["sentiment"],
        "reward": info["reward"],
        "tokens": ", ".join(info["tokens"])
    })

feedback_df = pd.DataFrame(feedback_rows)
display(feedback_df)

print("💡 INSIGHT: la boucle RL ne remplace pas le conseiller, elle apprend progressivement quels produits sont les plus pertinents par persona.")
print("💡 INSIGHT: le reward combine l'acceptation commerciale et la tonalité réelle du feedback terrain.")

In [ ]:
def generate_fiche_visite(client_id):
    if client_id not in df_final_features.index:
        raise KeyError(f"Client inconnu: {client_id}")

    row = df_final_features.loc[client_id]
    recommendations = recommend_products_for_client(client_id, top_n=3)
    lookalike_products = [
        row.get("Produit_Manquant_Secteur_1", ""),
        row.get("Produit_Manquant_Secteur_2", ""),
        row.get("Produit_Manquant_Secteur_3", ""),
    ]
    lookalike_products = [p for p in lookalike_products if p]

    lines = []
    lines.append(f"FICHE DE VISITE - Client {client_id}")
    lines.append(f"Persona IA: {row.get('Persona_Name', 'N/A')}")
    lines.append(f"Secteur: {row.get("Secteur d'activité", 'N/A')}")
    lines.append(f"Wilaya: {row.get('Wilaya', 'N/A')}")
    lines.append(f"PNB NET 15M: {row.get('PNB_NET_15m', 0):,.0f}")
    lines.append(f"Flux crediteurs 3M actuels: {row.get('Flux_Crediteurs_Current_3M', 0):,.0f}")
    lines.append(f"Flux crediteurs 3M precedents: {row.get('Flux_Crediteurs_Previous_3M', 0):,.0f}")
    lines.append(f"Taux evolution flux: {row.get('Taux_Evolution_Flux', 1):.2f}")
    lines.append(f"Score digital actif: {int(row.get('Score_Digital_Actif', 0))}")
    lines.append(f"Taux utilisation credit: {row.get('Taux_Utilisation_Credit', 0):.2f}")
    lines.append("")

    if int(row.get("Churn_Alert_Flag", 0)) == 1:
        lines.append("ALERTE CHURN: Baisse critique (>30%) des flux crediteurs detectee. Contacter le client en urgence pour un point de situation.")
        lines.append("")

    if lookalike_products:
        lines.append("Top 3 produits manquants du profil sectoriel: " + ", ".join(lookalike_products))
        lines.append(
            f"Argument Look-alike: Les entreprises de votre taille dans le secteur {row.get("Secteur d'activité", 'N/A')} utilisent massivement {lookalike_products[0]}. Nous vous proposons cette solution pour rester competitif."
        )
        lines.append("")

    if int(row.get("No_New_Credit_Allowed", 0)) == 1:
        lines.append("Regle anti-spam: ne pas proposer de nouveau credit tant que la ligne existante reste sous-utilisee (<30%).")
        lines.append("")

    if recommendations:
        lines.append("Recommandations priorisees:")
        for rank, reco in enumerate(recommendations, start=1):
            lines.append(
                f"{rank}. {reco['product']} ({reco['family']}) | score final={reco['score']:.2%} | model={reco['model_score']:.2%} | similarite={reco['content_score']:.2%} | secteur={reco['sector_score']:.2%}"
            )
    else:
        lines.append("Recommandations priorisees: aucune recommandation activable a ce stade.")

    return "\n".join(lines)

sample_client_id = (
    df_final_features.sort_values(["Churn_Alert_Flag", "PNB_NET_15m"], ascending=[False, False])
    .index[0]
)

sample_fiche = generate_fiche_visite(sample_client_id)
print(sample_fiche)
print("")
print("💡 INSIGHT: la NLG est déterministe, audit-able, et maintenant centrée sur un vrai Top 3 de produits manquants du profil client.")

In [ ]:
def format_amount(value):
    try:
        return f"{float(value):,.0f}".replace(",", " ")
    except Exception:
        return str(value)

def format_ratio(value):
    try:
        return f"{float(value):.2f}"
    except Exception:
        return str(value)

def export_client_pdf(client_id, output_dir=OUTPUT_DIR):
    row = df_final_features.loc[client_id]
    fiche_text = generate_fiche_visite(client_id)
    recommendations = recommend_products_for_client(client_id, top_n=3)

    pdf_path = output_dir / f"SGA_Fiche_Client_{client_id}.pdf"
    doc = SimpleDocTemplate(
        str(pdf_path),
        pagesize=A4,
        leftMargin=1.3 * cm,
        rightMargin=1.3 * cm,
        topMargin=1.0 * cm,
        bottomMargin=1.0 * cm
    )

    styles = getSampleStyleSheet()
    title_style = ParagraphStyle(
        "sga_title",
        parent=styles["Title"],
        fontName="Helvetica-Bold",
        fontSize=16,
        leading=18,
        alignment=TA_CENTER,
        textColor=colors.HexColor(SGA_NAVY),
        spaceAfter=6,
    )
    subtitle_style = ParagraphStyle(
        "sga_subtitle",
        parent=styles["BodyText"],
        fontName="Helvetica-Bold",
        fontSize=9,
        leading=11,
        alignment=TA_CENTER,
        textColor=colors.HexColor(SGA_RED),
        spaceAfter=6,
    )
    section_style = ParagraphStyle(
        "sga_section",
        parent=styles["Heading3"],
        fontName="Helvetica-Bold",
        fontSize=10,
        leading=12,
        alignment=TA_LEFT,
        textColor=colors.HexColor(SGA_NAVY),
        spaceAfter=4,
        spaceBefore=4,
    )
    body_style = ParagraphStyle(
        "sga_body",
        parent=styles["BodyText"],
        fontName="Helvetica",
        fontSize=8.5,
        leading=10.5,
        alignment=TA_LEFT,
        textColor=colors.black,
        spaceAfter=2,
    )

    elements = []

    logo_placeholder = Table([["LOGO SGA - Placeholder"]], colWidths=[6.0 * cm], rowHeights=[0.8 * cm], hAlign="CENTER")
    logo_placeholder.setStyle(TableStyle([
        ("BOX", (0, 0), (-1, -1), 0.8, colors.HexColor(SGA_GREY)),
        ("ALIGN", (0, 0), (-1, -1), "CENTER"),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("TEXTCOLOR", (0, 0), (-1, -1), colors.HexColor(SGA_GREY)),
        ("FONTNAME", (0, 0), (-1, -1), "Helvetica"),
        ("FONTSIZE", (0, 0), (-1, -1), 8),
    ]))
    elements.extend([
        logo_placeholder,
        Spacer(1, 0.15 * cm),
        Paragraph(pdf_safe("SGA - Sales Intelligence & Retention Engine"), title_style),
        Paragraph(pdf_safe(f"Fiche 360 - Client {client_id} | Persona {row.get('Persona_Name', 'N/A')}"), subtitle_style),
    ])

    identity_data = [
        ["Client", pdf_safe(client_id), "Segment", pdf_safe(row.get("Segmenattion", "N/A"))],
        ["Secteur", pdf_safe(row.get("Secteur d'activité", "N/A")), "Wilaya", pdf_safe(row.get("Wilaya", "N/A"))],
        ["Taille", pdf_safe(row.get("Client_Size", "N/A")), "Digital actif", str(int(row.get("Score_Digital_Actif", 0)))],
    ]
    identity_table = Table(identity_data, colWidths=[2.4 * cm, 6.0 * cm, 2.4 * cm, 4.0 * cm], hAlign="CENTER")
    identity_table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, -1), colors.HexColor(SGA_LIGHT)),
        ("BOX", (0, 0), (-1, -1), 0.6, colors.HexColor(SGA_GREY)),
        ("GRID", (0, 0), (-1, -1), 0.35, colors.HexColor(SGA_GREY)),
        ("FONTNAME", (0, 0), (-1, -1), "Helvetica"),
        ("FONTNAME", (0, 0), (-1, -1), "Helvetica"),
        ("FONTNAME", (0, 0), (0, -1), "Helvetica-Bold"),
        ("FONTNAME", (2, 0), (2, -1), "Helvetica-Bold"),
        ("ALIGN", (0, 0), (-1, -1), "CENTER"),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("FONTSIZE", (0, 0), (-1, -1), 8.4),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 5),
        ("TOPPADDING", (0, 0), (-1, -1), 5),
    ]))
    elements.extend([identity_table, Spacer(1, 0.2 * cm)])

    kpi_data = [
        ["KPI", "Valeur"],
        ["PNB NET 15M", format_amount(row.get("PNB_NET_15m", 0))],
        ["Flux crediteurs 3M actuels", format_amount(row.get("Flux_Crediteurs_Current_3M", 0))],
        ["Flux crediteurs 3M precedents", format_amount(row.get("Flux_Crediteurs_Previous_3M", 0))],
        ["Taux evolution flux", format_ratio(row.get("Taux_Evolution_Flux", 1))],
        ["Taux utilisation credit", format_ratio(row.get("Taux_Utilisation_Credit", 0))],
        ["Churn alert", str(int(row.get("Churn_Alert_Flag", 0)))],
        ["No New Credit Allowed", str(int(row.get("No_New_Credit_Allowed", 0)))],
    ]
    kpi_table = Table(kpi_data, colWidths=[8.0 * cm, 4.0 * cm], hAlign="CENTER")
    kpi_style_commands = [
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor(SGA_NAVY)),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
        ("ALIGN", (0, 0), (-1, -1), "CENTER"),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("GRID", (0, 0), (-1, -1), 0.45, colors.HexColor(SGA_GREY)),
        ("BOX", (0, 0), (-1, -1), 0.7, colors.HexColor(SGA_GREY)),
        ("FONTSIZE", (0, 0), (-1, -1), 8.6),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 5),
        ("TOPPADDING", (0, 0), (-1, -1), 5),
    ]
    for row_idx in range(1, len(kpi_data)):
        bg = colors.whitesmoke if row_idx % 2 == 1 else colors.HexColor("#EEF3F8")
        kpi_style_commands.append(("BACKGROUND", (0, row_idx), (-1, row_idx), bg))
    kpi_table.setStyle(TableStyle(kpi_style_commands))
    elements.extend([Paragraph("Tableau KPI centré", section_style), kpi_table, Spacer(1, 0.2 * cm)])

    reco_data = [["Rang", "Produit manquant", "Famille", "Score final"]]
    if recommendations:
        for rank, reco in enumerate(recommendations, start=1):
            reco_data.append([
                str(rank),
                pdf_safe(reco["product"]),
                pdf_safe(reco["family"]),
                f"{reco['score']:.2%}",
            ])
    else:
        reco_data.append(["-", "Aucune recommandation activable", "-", "-"])

    reco_table = Table(reco_data, colWidths=[1.3 * cm, 6.7 * cm, 3.2 * cm, 2.6 * cm], hAlign="CENTER")
    reco_style_commands = [
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor(SGA_RED)),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
        ("ALIGN", (0, 0), (-1, -1), "CENTER"),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("GRID", (0, 0), (-1, -1), 0.45, colors.HexColor(SGA_GREY)),
        ("BOX", (0, 0), (-1, -1), 0.7, colors.HexColor(SGA_GREY)),
        ("FONTSIZE", (0, 0), (-1, -1), 8.1),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 4),
        ("TOPPADDING", (0, 0), (-1, -1), 4),
    ]
    for row_idx in range(1, len(reco_data)):
        bg = colors.HexColor("#FFF4F6") if row_idx % 2 == 1 else colors.white
        reco_style_commands.append(("BACKGROUND", (0, row_idx), (-1, row_idx), bg))
    reco_table.setStyle(TableStyle(reco_style_commands))
    elements.extend([Paragraph("Top 3 recommandations", section_style), reco_table, Spacer(1, 0.15 * cm)])

    fiche_lines = [line for line in fiche_text.split("\n") if line.strip()]
    fiche_paragraphs = [Paragraph(pdf_safe(line), body_style) for line in fiche_lines]
    elements.extend([Paragraph("Fiche de visite / argumentaire", section_style)] + fiche_paragraphs)

    doc.build(elements)
    return pdf_path

client_ai_insights = {}
for client_id in df_final_features.index:
    client_ai_insights[str(client_id)] = {
        "persona": str(df_final_features.loc[client_id, "Persona_Name"]),
        "sector": str(df_final_features.loc[client_id, "Secteur d'activité"]),
        "wilaya": str(df_final_features.loc[client_id, "Wilaya"]),
        "kpis": {
            "PNB_NET_15m": float(df_final_features.loc[client_id, "PNB_NET_15m"]),
            "Flux_Crediteurs_Current_3M": float(df_final_features.loc[client_id, "Flux_Crediteurs_Current_3M"]),
            "Flux_Crediteurs_Previous_3M": float(df_final_features.loc[client_id, "Flux_Crediteurs_Previous_3M"]),
            "Taux_Evolution_Flux": float(df_final_features.loc[client_id, "Taux_Evolution_Flux"]),
            "Churn_Alert_Flag": int(df_final_features.loc[client_id, "Churn_Alert_Flag"]),
            "Taux_Utilisation_Credit": float(df_final_features.loc[client_id, "Taux_Utilisation_Credit"]),
            "No_New_Credit_Allowed": int(df_final_features.loc[client_id, "No_New_Credit_Allowed"]),
            "Score_Digital_Actif": int(df_final_features.loc[client_id, "Score_Digital_Actif"]),
            "Ecart_Moyenne_Secteur": float(df_final_features.loc[client_id, "Ecart_Moyenne_Secteur"]),
        },
        "top_missing_profile_products": [
            str(df_final_features.loc[client_id, "Produit_Manquant_Secteur_1"]),
            str(df_final_features.loc[client_id, "Produit_Manquant_Secteur_2"]),
            str(df_final_features.loc[client_id, "Produit_Manquant_Secteur_3"]),
        ],
        "recommendations": recommend_products_for_client(client_id, top_n=3),
        "fiche_visite": generate_fiche_visite(client_id),
    }

csv_path = OUTPUT_DIR / "df_final_features.csv"
json_path = OUTPUT_DIR / "client_ai_insights.json"
metrics_path = OUTPUT_DIR / "model_metrics.json"
model_bundle_path = OUTPUT_DIR / "sga_sales_engine_bundle.pkl"
model_manifest_path = OUTPUT_DIR / "exported_model_manifest.json"
catboost_export_dir = OUTPUT_DIR / "catboost_models"
catboost_export_dir.mkdir(exist_ok=True)

product_model_manifest = {}
for product_name, info in product_models.items():
    export_name = None
    if info is not None and info.get("model") is not None:
        export_name = f"catboost_{slugify_text(product_label_lookup.get(product_name, product_name))}.cbm"
        info["model"].save_model(str(catboost_export_dir / export_name))
    product_model_manifest[product_name] = {
        "product_label": product_label_lookup.get(product_name, product_name),
        "family": product_family_lookup.get(product_name, "Other"),
        "has_model": bool(info is not None and info.get("model") is not None),
        "export_file": export_name,
        "feature_columns": info.get("feature_columns", []) if info is not None else [],
    }

model_bundle = {
    "product_catalog_df": product_catalog_df.to_dict(orient="records"),
    "recommendable_products": product_cols,
    "credit_products": credit_product_cols,
    "product_label_lookup": product_label_lookup,
    "product_family_lookup": product_family_lookup,
    "cluster_scaler": cluster_scaler,
    "gmm_model": gmm,
    "content_encoder": content_encoder,
    "content_knn": content_knn,
    "model_numeric_cols": model_numeric_cols,
    "model_categorical_cols": model_categorical_cols,
    "model_metrics": model_metrics,
    "product_model_manifest": product_model_manifest,
}

df_final_features.reset_index().to_csv(csv_path, index=False, encoding="utf-8-sig")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(client_ai_insights, f, ensure_ascii=False, indent=2)
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(model_metrics, f, ensure_ascii=False, indent=2)
with open(model_manifest_path, "w", encoding="utf-8") as f:
    json.dump(product_model_manifest, f, ensure_ascii=False, indent=2)
with open(model_bundle_path, "wb") as f:
    pickle.dump(model_bundle, f)

sample_pdf_path = export_client_pdf(sample_client_id)

print(f"✅ CSV exporté : {csv_path}")
print(f"✅ JSON exporté : {json_path}")
print(f"✅ Metrics exporté : {metrics_path}")
print(f"✅ Bundle modèle exporté : {model_bundle_path}")
print(f"✅ Manifest modèles exporté : {model_manifest_path}")
print(f"✅ Répertoire CatBoost : {catboost_export_dir}")
print(f"✅ PDF exemple  : {sample_pdf_path}")
print("💡 INSIGHT: les livrables incluent désormais un export complet du moteur et un PDF plus lisible, avec KPI centrés sous forme de tableau.")

## 🧭 Dashboard Roadmap — Recommandation pour la DSI SGA

### Front-end
- **React** pour la structure applicative
- **Tremor UI** pour les cartes KPI, segments, recommandations et alertes churn
- **TanStack Query** pour la gestion propre des appels API

### Back-end
- **FastAPI** pour exposer :
  - `/clients/{id}/fiche`
  - `/clients/{id}/recommendations`
  - `/clients/{id}/risk`
  - `/feedback/reward`
- **Pydantic** pour la validation forte des payloads

### Data & Storage
- **PostgreSQL** pour stocker :
  - features clients
  - historique recommandations
  - feedback conseillers
  - rewards RL
- **Parquet/CSV** pour les batchs analytiques
- **Airflow** ou cron industriel pour les recalculs périodiques

### Monitoring
- Suivi du taux d’acceptation des recommandations
- Suivi du churn observé vs churn prédit
- Drift sectoriel et drift de distribution des produits
- Monitoring des performances CatBoost par produit

### Architecture cible
1. Batch mensuel ou hebdomadaire
2. Recalcul features + segments + scores
3. Exposition FastAPI
4. Dashboard React/Tremor pour les Chargés d’Affaires
5. Boucle feedback → NLP local → RL bandit → amélioration continue